# Localization Sample : Kalman filter (EKF)

---- 

- conda env : [ai_robotics](../../README.md#setup-a-conda-environment)

---

### Ref
- https://github.com/AtsushiSakai/PythonRobotics/


| Feature            | [EKF](./extended_kalman_filter.ipynb) | [KF](./kalman_filter.ipynb)      |
| ------------------ | -------------------------------------- | ------------------------------- |
| Motion model       | Nonlinear                              | Linear                          |
| Uses Jacobians     | ✅ Yes (`jacob_f`, `jacob_h`)           | ❌ No                            |
| System matrix F    | Varies with yaw                        | Constant or mildly linearized   |
| Suitable for       | Nonlinear motion (e.g., turning robot) | Linear/constant velocity motion |
| Computational cost | Higher                                 | Lower                           |


### Imports and Configuration

In [1]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[0])
# Add this path to sys.path
sys.path.insert(0, parent_dir)


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import math
from utils.plot import plot_covariance_ellipse

# Simulation parameters
DT = 0.1        # time step [s]
SIM_TIME = 50.0 # total simulation time [s]

# Process and observation noise
Q = np.diag([0.1, 0.1, np.deg2rad(1.0), 1.0]) ** 2   # Process noise covariance
R = np.diag([0.5, 0.5]) ** 2                         # Observation noise covariance

# Motion noise (for simulating real world)
INPUT_NOISE = np.diag([1.0, np.deg2rad(10.0)]) ** 2
GPS_NOISE = np.diag([0.5, 0.5]) ** 2


### Motion and Functions

In [5]:
def calc_input():
    """Control input: constant forward motion and yaw rate."""
    v = 1.0       # m/s
    yawrate = 0.1 # rad/s
    return np.array([[v], [yawrate]])

def motion_model(x, u):
    """Linearized motion model for constant velocity + yawrate system."""
    F = np.array([[1.0, 0, -u[0,0] * DT * math.sin(x[2,0])],
                  [0, 1.0,  u[0,0] * DT * math.cos(x[2,0])],
                  [0, 0, 1.0]])
    B = np.array([[DT * math.cos(x[2,0]), 0],
                  [DT * math.sin(x[2,0]), 0],
                  [0.0, DT]])
    return F, B

def true_motion_model(x, u):
    """Used for simulating the real trajectory (no linearization)."""
    x[0,0] += u[0,0] * DT * math.cos(x[2,0])
    x[1,0] += u[0,0] * DT * math.sin(x[2,0])
    x[2,0] += u[1,0] * DT
    return x

def observation_model(x):
    """Observation model: directly measure x and y position."""
    H = np.array([[1, 0, 0],
                  [0, 1, 0]])
    z = H @ x
    return z, H

def kalman_filter(xEst, PEst, z, u):
    """Standard Kalman Filter update step for linear system."""
    # Predict
    F, B = motion_model(xEst, u)
    xPred = F @ xEst + B @ u
    PPred = F @ PEst @ F.T + Q[:3,:3]  # only 3x3 portion for position/yaw

    # Update
    zPred, H = observation_model(xPred)
    y = z - zPred
    S = H @ PPred @ H.T + R
    K = PPred @ H.T @ np.linalg.inv(S)
    xEst = xPred + K @ y
    PEst = (np.eye(len(xEst)) - K @ H) @ PPred

    return xEst, PEst

def simulate_observation(xTrue, u):
    """Simulate true motion and noisy observation."""
    xTrue = true_motion_model(xTrue, u)
    z = observation_model(xTrue)[0] + GPS_NOISE @ np.random.randn(2,1)
    return xTrue, z



### Animation Setup and Update Function

In [ ]:
def run_kf_animation():
    xEst = np.zeros((3,1))
    xTrue = np.zeros((3,1))
    PEst = np.eye(3)
    time = 0.0

    hxEst, hxTrue, hz = xEst, xTrue, np.zeros((2,1))

    fig, ax = plt.subplots(figsize=(6,6))
    ax.set_aspect('equal')
    ax.grid(True)

    def init():
        ax.cla()
        ax.grid(True)
        ax.set_aspect('equal')
        return []

    def update(frame):
        nonlocal xEst, xTrue, PEst, hxEst, hxTrue, hz, time
        time += DT
        if time > SIM_TIME:
            ani.event_source.stop()
            return []

        u = calc_input()
        xTrue, z = simulate_observation(xTrue, u)
        xEst, PEst = kalman_filter(xEst, PEst, z, u)

        hxEst = np.hstack((hxEst, xEst))
        hxTrue = np.hstack((hxTrue, xTrue))
        hz = np.hstack((hz, z))

        # Clean up frame
        ax.cla()
        ax.grid(True)
        ax.set_aspect('equal')

        # Auto-scale limits
        all_x = np.hstack((hxTrue[0,:], hxEst[0,:]))
        all_y = np.hstack((hxTrue[1,:], hxEst[1,:]))
        x_min, x_max = np.min(all_x), np.max(all_x)
        y_min, y_max = np.min(all_y), np.max(all_y)
        margin_x = (x_max - x_min)*0.2 + 0.5
        margin_y = (y_max - y_min)*0.2 + 0.5
        ax.set_xlim(x_min - margin_x, x_max + margin_x)
        ax.set_ylim(y_min - margin_y, y_max + margin_y)

        # Draw
        ax.plot(hz[0,:], hz[1,:], ".g", label="observation")
        ax.plot(hxTrue[0,:], hxTrue[1,:], "-b", label="true")
        ax.plot(hxEst[0,:], hxEst[1,:], "-r", label="KF estimate")
        plot_covariance_ellipse(xEst[0,0], xEst[1,0], PEst, ax=ax)

        ax.legend(loc="lower right", fontsize="small", frameon=True, shadow=True)
        ax.set_title(f"Kalman Filter Localization | Time = {time:.1f}s")

        return []

    ani = FuncAnimation(fig, update, frames=int(SIM_TIME / DT),
                        init_func=init, interval=100, blit=False, repeat=False)
    plt.close(fig)  # prevent duplicate static plot
    return ani

SIM_TIME = 3 # Update the duration of total simulation time [s]
ani = run_kf_animation()
HTML(ani.to_html5_video())
